# 英国道路碰撞严重程度预测

本 Notebook 展示英国 2021—2025 年已报告道路碰撞的三阶段分析，内容包括严重程度构成、时间与环境模式、空间差异，以及 KSI 预测模型的表现。

时间评估使用 2021—2023 年训练，在 2024 年选择模型和分类阈值，并将 2025 年作为独立测试年份。


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from IPython.display import display
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, confusion_matrix, precision_recall_curve

# Locate the repository whether the notebook is launched from the root or notebooks/.
START = Path.cwd().resolve()
ROOT = next((p for p in [START, *START.parents] if (p / "configs/default.yaml").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the project repository.")

sys.path.insert(0, str(ROOT / "src"))

from road_severity.analysis import column_decisions_table, raw_quality_table
from road_severity.data import (
    build_features,
    clean_collisions,
    coded_missing_summary,
    field_unification_summary,
    make_target,
    parse_temporal_fields,
    read_raw_collisions,
    resolve_duplicates,
    stratified_sample,
    validate_collisions,
    validate_schema_contract,
)
from road_severity.modeling import evaluate, make_pipeline, select_threshold, temporal_split
import road_severity.processed_analysis as processed_analysis
from road_severity.processed_analysis import (
    HEAT_RED,
    LABELS,
    build_all_tables,
    cross_summary,
    plot_annual,
    plot_heatmap,
    plot_hourly,
    plot_proportion,
    plot_severity,
    plot_spatial_hex,
)

CONFIG_PATH = ROOT / "configs/default.yaml"
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
print(f"项目根目录： {ROOT}")

## 阶段 1 — 数据验证与处理

原始数据按照 DfT 字段契约进行检查，处理完全重复记录，解析时间字段，并统一分类编码。死亡与重伤碰撞构成 KSI 目标；伤情结果衍生字段不进入建模。


In [ ]:
def process_raw_collisions(
    config: dict,
    raw_path: Path | None = None,
    max_rows: int | None = None,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    raw_path = raw_path or ROOT / config["data"]["raw_path"]
    schema_path = ROOT / config["data"]["schema_path"]
    contract = yaml.safe_load(schema_path.read_text(encoding="utf-8"))
    max_rows = config["project"]["max_rows"] if max_rows is None else max_rows

    raw = read_raw_collisions(raw_path)
    raw_quality = raw_quality_table(raw)
    contract_issues = validate_schema_contract(raw, contract)
    if contract_issues["failed_rows"].sum():
        display(contract_issues)
        raise RuntimeError("缺少数据契约要求的字段。")

    deduplicated, duplicate_summary, duplicate_conflicts = resolve_duplicates(raw)
    validation = pd.concat(
        [contract_issues, validate_collisions(deduplicated, contract)], ignore_index=True
    )
    conflict_check = pd.DataFrame([{
        "check": "conflicting_duplicate_collision_index",
        "severity": "error",
        "failed_rows": int(duplicate_conflicts["collision_index"].nunique()),
        "example_row_indices": ";".join(map(str, duplicate_conflicts.index.tolist()[:5])),
        "details": ";".join(map(str, duplicate_conflicts["collision_index"].drop_duplicates().tolist()[:10])),
    }])
    validation = pd.concat([validation, conflict_check], ignore_index=True)
    error_count = int(validation.loc[validation["severity"].eq("error"), "failed_rows"].sum())
    if error_count:
        display(validation)
        raise RuntimeError(f"数据验证发现 {error_count} 项错误级问题。")

    parsed = parse_temporal_fields(deduplicated)
    sampled = stratified_sample(
        parsed,
        None if max_rows == 0 else max_rows,
        config["project"]["random_state"],
    )
    processed = clean_collisions(sampled)
    processed_quality = raw_quality_table(processed)
    duplicates_removed = int(
        duplicate_summary.loc[
            duplicate_summary["measure"].eq("exact_duplicate_rows_removed"), "value"
        ].iloc[0]
    )
    processing_summary = pd.DataFrame({
        "measure": [
            "full_raw_rows", "sampled_rows", "processed_rows", "duplicates_removed",
            "raw_columns", "processed_columns", "years", "ksi_share",
        ],
        "value": [
            str(len(raw)), str(len(sampled)), str(len(processed)), str(duplicates_removed),
            str(len(raw.columns)), str(len(processed.columns)),
            str(processed["collision_year"].nunique()), f"{processed['ksi'].mean():.12f}",
        ],
    })
    audit_tables = {
        "raw_data_quality": raw_quality,
        "validation_issues": validation,
        "duplicate_summary": duplicate_summary,
        "duplicate_conflicts": duplicate_conflicts,
        "coded_missing_summary": coded_missing_summary(raw, contract),
        "field_unification_summary": field_unification_summary(
            sampled.reset_index(drop=True), processed
        ),
        "processing_summary": processing_summary,
        "processed_data_quality": processed_quality,
        "column_decisions": column_decisions_table(processed, processed_quality),
    }
    display(processing_summary)
    return processed, audit_tables



## 阶段 2 — 描述性分析

描述性分析比较六个核心模式：

1. 碰撞严重程度构成；
2. 年度碰撞数量与严重程度记录方式；
3. 分时段碰撞数量与 KSI 占比；
4. 不同照明条件下的 KSI 占比；
5. 限速与城乡环境之间的交互关系；
6. 空间碰撞密度与 KSI 占比。


In [ ]:
REQUIRED_PROCESSED_COLUMNS = {
    "collision_year", "collision_severity", "ksi", "date", "hour", "month",
    "collision_injury_based", "collision_adjusted_severity_serious",
    "speed_limit", "urban_or_rural_area", "light_conditions", "road_type",
    "weather_conditions", "road_surface_conditions", "day_of_week",
    "junction_detail_unified", "pedestrian_crossing_unified", "carriageway_hazards_unified",
    "junction_detail_unified_source", "pedestrian_crossing_unified_source",
    "carriageway_hazards_unified_source", "longitude", "latitude",
}


def validate_processed_input(frame: pd.DataFrame, metadata: dict, contract: dict) -> pd.DataFrame:
    checks = [
        ("metadata_requires_full_data", metadata.get("sampling_strategy") == "full_data", metadata.get("sampling_strategy")),
        ("metadata_row_count_matches_file", metadata.get("processed_rows") == len(frame), f"metadata={metadata.get('processed_rows')}; actual={len(frame)}"),
        ("metadata_contract_version_matches", metadata.get("data_contract_version") == contract.get("version"), f"metadata={metadata.get('data_contract_version')}; expected={contract.get('version')}"),
        ("required_processed_columns_present", REQUIRED_PROCESSED_COLUMNS.issubset(frame.columns), ";".join(sorted(REQUIRED_PROCESSED_COLUMNS.difference(frame.columns)))),
        ("processed_data_is_not_empty", len(frame) > 0, str(len(frame))),
    ]
    return pd.DataFrame([
        {"check": name, "passed": passed, "details": details}
        for name, passed, details in checks
    ])


processed_path = ROOT / config["data"]["processed_path"]
metadata_path = processed_path.parent / "processing_metadata.json"
if not processed_path.exists() or not metadata_path.exists():
    raise FileNotFoundError("缺少项目中的处理后数据或元数据。")

metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
contract = yaml.safe_load((ROOT / config["data"]["schema_path"]).read_text(encoding="utf-8"))
frame = pd.read_csv(processed_path, low_memory=False, parse_dates=["date"])

processed_validation = validate_processed_input(frame, metadata, contract)
display(processed_validation)
if not processed_validation["passed"].all():
    raise RuntimeError("阶段 2 输入验证失败。")

analysis_tables = build_all_tables(frame)

print(f"数据范围：{frame['collision_year'].min()}—{frame['collision_year'].max()} 年，共 {len(frame):,} 条已报告碰撞。")

In [ ]:
# 应用阶段 2 的统一图表布局。
STAGE_2_TEXT_ZH = {
    "Three in four reported collisions were recorded as slight": "约四分之三的报告碰撞被记录为轻微碰撞",
    "Share of reported collisions (%)": "报告碰撞占比（%）",
    "Collision volume stayed stable while injury-based reporting expanded": "碰撞总量保持稳定，基于伤情的严重程度记录方式持续扩大",
    "Year": "年份", "Reported collisions": "报告碰撞数",
    "Injury-based severity reporting (%)": "基于伤情的严重程度记录占比（%）",
    "Collision volume peaks in the afternoon, but severity peaks overnight": "碰撞数量在下午达到峰值，严重程度占比则在深夜最高",
    "Hour of day": "一天中的小时", "KSI collisions (% of reported collisions)": "KSI 碰撞（占报告碰撞的比例）",
    "Speed-limit patterns differ substantially between urban and rural roads": "不同限速下的严重程度模式在城乡道路之间差异明显",
    "Longitude": "经度", "Latitude": "纬度",
    "Severity hotspots do not simply follow collision density": "严重碰撞热点并不只是随碰撞密度变化",
}


def show_processed_figure(
    fig: plt.Figure,
    _path=None,
    footer: str | None = None,
    apply_tight_layout: bool = True,
) -> None:
    for ax in fig.axes:
        ax.set_title(STAGE_2_TEXT_ZH.get(ax.get_title(), ax.get_title()))
        ax.set_xlabel(STAGE_2_TEXT_ZH.get(ax.get_xlabel(), ax.get_xlabel()))
        ax.set_ylabel(STAGE_2_TEXT_ZH.get(ax.get_ylabel(), ax.get_ylabel()))
    source_zh = "数据来源：英国交通部道路碰撞数据，2021—2025 年。"
    footer = source_zh
    if footer:
        fig.text(0.01, 0.008, footer, ha="left", va="bottom", fontsize=8, color=processed_analysis.DARK_GREY)
    if apply_tight_layout:
        fig.tight_layout(rect=(0, 0.035, 1, 1) if footer else None)
    else:
        fig.subplots_adjust(left=0.08, right=0.94, bottom=0.08, top=0.92)
    display(fig)
    plt.close(fig)


processed_analysis._save = show_processed_figure
processed_analysis._style()

plot_severity(analysis_tables["severity_summary"], None)
plot_annual(analysis_tables["annual_reporting_summary"], None)
plot_hourly(analysis_tables["hourly_summary"], None)
plot_proportion(
    analysis_tables["ksi_by_light"],
    None,
    "无照明黑暗环境下记录的 KSI 占比最高",
)

area_labels = {1: "Urban", 2: "Rural"}
speed_labels = {
    20: "20 mph", 30: "30 mph", 40: "40 mph",
    50: "50 mph", 60: "60 mph", 70: "70 mph",
}
speed_area = cross_summary(
    frame[frame["urban_or_rural_area"].isin([1, 2])],
    "urban_or_rural_area", "speed_limit", area_labels, speed_labels,
)
plot_heatmap(
    speed_area,
    None,
    "不同限速下的严重程度模式在城乡道路之间差异明显",
    base_color=HEAT_RED,
)

import io
spatial_summary_buffer = io.StringIO()
plot_spatial_hex(frame, None, spatial_summary_buffer)



## 阶段 3 — 模型选择与独立测试

在 2024 年验证集上比较多数类基线、类别加权逻辑回归和调参后的 LightGBM，并使用同一验证集确定分类阈值，最后在独立的 2025 年测试集上进行评估。


In [ ]:
best_params_path = ROOT / config["model"]["best_params_path"]
if not best_params_path.exists():
    raise FileNotFoundError(
        f"Best LightGBM parameters not found: {best_params_path}. Run scripts/tune_lightgbm.py first."
    )

best_params = yaml.safe_load(best_params_path.read_text(encoding="utf-8"))["lightgbm"]
model_settings = {**config["model"], **best_params}
train, validation, test = temporal_split(
    frame, config["model"]["validation_year"], config["model"]["test_year"]
)
X_train, y_train = build_features(train), make_target(train, config["model"]["task"])
X_valid, y_valid = build_features(validation), make_target(validation, config["model"]["task"])
X_test, y_test = build_features(test), make_target(test, config["model"]["task"])

comparison = []
fitted_models = {}
for kind in ["dummy", "logistic_regression", "lightgbm"]:
    candidate = make_pipeline(
        X_train, config["project"]["random_state"], model_settings, kind
    )
    candidate.fit(X_train, y_train)
    validation_metrics = evaluate(candidate, X_valid, y_valid)
    comparison.append({
        "model": kind,
        "roc_auc": validation_metrics["roc_auc"],
        "average_precision": validation_metrics["average_precision"],
        "brier_score": validation_metrics["brier_score"],
    })
    fitted_models[kind] = candidate

comparison_frame = pd.DataFrame(comparison).sort_values("average_precision", ascending=False)
selected_kind = comparison_frame.iloc[0]["model"]
selected_model = fitted_models[selected_kind]
validation_probabilities = selected_model.predict_proba(X_valid)[:, 1]
threshold = select_threshold(y_valid, validation_probabilities)
test_metrics = evaluate(selected_model, X_test, y_test, threshold)

display(comparison_frame.style.format({
    "roc_auc": "{:.3f}", "average_precision": "{:.3f}", "brier_score": "{:.3f}"
}))
print(f"入选模型： {selected_kind}; 验证集确定的阈值： {threshold:.2f}")
print({key: round(value, 3) for key, value in test_metrics.items() if isinstance(value, float)})


importance_rows = 10_000
if importance_rows and len(X_test) > importance_rows:
    sampled_X = X_test.sample(importance_rows, random_state=config["project"]["random_state"])
    sampled_y = y_test.loc[sampled_X.index]
else:
    sampled_X, sampled_y = X_test, y_test

importance_result = permutation_importance(
    selected_model,
    sampled_X,
    sampled_y,
    n_repeats=5,
    scoring="average_precision",
    random_state=config["project"]["random_state"],
    n_jobs=1,
)
importance = pd.DataFrame({
    "feature": sampled_X.columns,
    "importance_mean": importance_result.importances_mean,
    "importance_std": importance_result.importances_std,
}).sort_values("importance_mean", ascending=False)


In [ ]:
# 模型评估图。
MODEL_ACCENT = "#39728C"
MATRIX_ACCENT = "#8064A2"
MODEL_GREY = "#B8BDC2"
MODEL_DARK_GREY = "#555B61"
MODEL_GRID_GREY = "#E5E7E9"
MODEL_SOURCE = "数据来源：英国交通部道路碰撞数据；独立的 2025 年测试集。"


def set_model_figure_style() -> None:
    sns.set_theme(style="whitegrid", rc={
        "axes.edgecolor": MODEL_DARK_GREY,
        "axes.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": MODEL_GRID_GREY,
        "grid.linewidth": 0.7,
        "axes.titleweight": "normal",
        "figure.facecolor": "white",
    })


def show_model_figure(fig: plt.Figure) -> None:
    fig.text(0.01, 0.008, MODEL_SOURCE, ha="left", va="bottom", fontsize=8, color=MODEL_DARK_GREY)
    fig.tight_layout(rect=(0, 0.035, 1, 1))
    display(fig)
    plt.close(fig)


def create_story_model_figures(
    model,
    X: pd.DataFrame,
    y: pd.Series,
    threshold: float,
    importance: pd.DataFrame,
) -> None:
    set_model_figure_style()
    probabilities = model.predict_proba(X)[:, 1]
    predictions = (probabilities >= threshold).astype(int)
    prevalence = float(y.mean())

    precision, recall, _ = precision_recall_curve(y, probabilities)
    average_precision = average_precision_score(y, probabilities)
    fig, ax = plt.subplots(figsize=(6.5, 5))
    ax.plot(recall, precision, color=MODEL_ACCENT, linewidth=2)
    ax.axhline(prevalence, linestyle="--", color=MODEL_DARK_GREY)
    ax.set(
        title=f"平均精确率为 {average_precision:.3f}，高于基准发生率 {prevalence:.3f}",
        xlabel="召回率（被识别的 KSI 碰撞占比）",
        ylabel="精确率（预警中实际为 KSI 的占比）",
        xlim=(0, 1), ylim=(0, 1),
    )
    ax.annotate(
        f"基准发生率 {prevalence:.1%}", (0.98, prevalence), xytext=(6, 7),
        textcoords="offset points", color=MODEL_DARK_GREY, fontsize=9, va="center",
    )
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    show_model_figure(fig)

    matrix = confusion_matrix(y, predictions)
    row_share = matrix / matrix.sum(axis=1, keepdims=True)
    labels = np.array([
        [f"{matrix[i, j]:,}\n{row_share[i, j]:.1%} of actual class" for j in range(2)]
        for i in range(2)
    ])
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        row_share,
        annot=labels,
        fmt="",
        cmap=sns.light_palette(MATRIX_ACCENT, as_cmap=True),
        vmin=0,
        vmax=1,
        cbar=False,
        square=True,
        xticklabels=["预测为轻微", "预测为 KSI"],
        yticklabels=["实际为轻微", "实际为 KSI"],
        ax=ax,
    )
    ax.set(
        title=f"阈值为 {threshold:.2f} 时，模型识别出 {row_share[1, 1]:.1%} 的 KSI 碰撞",
        xlabel="预测类别", ylabel="实际类别",
    )
    show_model_figure(fig)

    observed, predicted = calibration_curve(y, probabilities, n_bins=10, strategy="quantile")
    max_gap = float(np.max(np.abs(observed - predicted)))
    upper = min(1.0, max(observed.max(), predicted.max()) * 1.12)
    fig, ax = plt.subplots(figsize=(6.5, 5))
    ax.plot(predicted, observed, marker="o", color=MODEL_ACCENT, linewidth=2)
    ax.plot([0, upper], [0, upper], "--", color=MODEL_DARK_GREY)
    ax.set(
        title=f"各风险组的最大校准偏差为 {max_gap:.1%}",
        xlabel="平均预测 KSI 概率", ylabel="实际 KSI 占比",
        xlim=(0, upper), ylim=(0, upper),
    )
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    show_model_figure(fig)

    top = importance.head(15).sort_values("importance_mean")
    top = top.assign(feature_label=top["feature"].str.replace("_", " ").str.title())
    highlight = top["importance_mean"].idxmax()
    colors = [MODEL_ACCENT if index == highlight else MODEL_GREY for index in top.index]
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(
        top["feature_label"], top["importance_mean"], xerr=top["importance_std"],
        color=colors, ecolor=MODEL_DARK_GREY, capsize=2,
    )
    most_important = top.loc[highlight]
    label_x = most_important["importance_mean"] + most_important["importance_std"] + 0.001
    ax.text(
        label_x, most_important["feature_label"], f"{most_important['importance_mean']:.3f}",
        va="center", ha="left", fontsize=9, color=MODEL_DARK_GREY,
    )
    ax.set_xlim(0, float((top["importance_mean"] + top["importance_std"]).max()) * 1.20)
    ax.set(
        title=f"{top.iloc[-1]['feature_label']} 对测试年预测信息的贡献最大",
        xlabel="特征置换后平均精确率的下降幅度", ylabel="特征",
    )
    ax.grid(axis="y", visible=False)
    show_model_figure(fig)



In [ ]:
create_story_model_figures(
    selected_model, X_test, y_test, threshold, importance
)
